# Phase 1: Loading, Exploring and Cleaning the Datasets

In [1]:
import pandas as pd

# Loading data
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')

# Merging on TransactionID
train = pd.merge(train_trans, train_id, on='TransactionID', how='left')

print("Data successfully loaded and merged...")
print(f"Total rows: {len(train)}")
print(f"Train dataset shape: {train.shape}")

Data successfully loaded and merged...
Total rows: 590540
Train dataset shape: (590540, 434)


In [2]:
# Calculating the percentage of missing values for each feature
missing_percentages = (train.isnull().sum() / len(train)) * 100

# Sorting the results in descending order
missing_percentages = missing_percentages.sort_values(ascending=False)

# Top 30 features with most missing values
print('Top 30 features with most missing values: ')
print(missing_percentages.head(30))

Top 30 features with most missing values: 
id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
id_09    87.312290
D8       87.312290
id_10    87.312290
D9       87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
V138     86.123717
V139     86.123717
V148     86.123717
dtype: float64


In [3]:
# Dropping features with more than 99% of missing data
features_to_drop = missing_percentages[missing_percentages > 99].index

# Drop those columns
train = train.drop(columns=features_to_drop)
print(f"Total features dropped: {len(features_to_drop)}")
print(f"New Train dataset shape: {train.shape}")

Total features dropped: 9
New Train dataset shape: (590540, 425)


# Phase 2: Feature Engineering


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

print("Starting memory-optimized feature engineering pipeline...")

# Initializing a blank dictionary to collect columns in background memory
new_features = {}

# Computing Cyclical Time Features
new_features['hour_of_day'] = (train['TransactionDT'] // 3600) % 24
new_features['day_of_week'] = (train['TransactionDT'] // (3600 * 24)) % 7

# Setting up Automated Sorting for Encodings
categorical_cols = train.select_dtypes(include=['object', 'str', 'category']).columns
target_encode_cols = []
label_encode_cols = []

for col in categorical_cols:
    unique_count = train[col].nunique()
    if 100 < unique_count <= 1000:
        target_encode_cols.append(col)
    elif unique_count <= 100:
        label_encode_cols.append(col)

# Execute Leakage-Free K-Fold Target Encoding using raw NumPy structures
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_encode_cols:
    new_features[f'{col}_target_enc'] = np.empty(len(train))
    new_features[f'{col}_target_enc'][:] = np.nan

for train_idx, val_idx in kf.split(train):
    X_train_fold = train.iloc[train_idx]
    X_val_fold = train.iloc[val_idx]

    for col in target_encode_cols:
        col_probabilities = X_train_fold.groupby(col)['isFraud'].mean()
        new_features[f'{col}_target_enc'][val_idx] = X_val_fold[col].map(col_probabilities)

# Fill unmapped rare categories with the global average fraud rate
global_mean = train['isFraud'].mean()
for col in target_encode_cols:
    arr = new_features[f'{col}_target_enc']
    arr[np.isnan(arr)] = global_mean
    new_features[f'{col}_target_enc'] = arr

# 5. Execute Label Encoding
for col in label_encode_cols:
    new_features[f'{col}_label_enc'] = train[col].astype('category').cat.codes

# 6. Compute Group Aggregations
new_features['card1_TransactionAmt_mean'] = train.groupby('card1')['TransactionAmt'].transform('mean')

# 7. Single-Step Concatenation: Build the new table and attach it side-by-side
new_features_df = pd.DataFrame(new_features, index=train.index)
train = pd.concat([train, new_features_df], axis=1)

print("All features engineered successfully with zero memory fragmentation!")
print(f"Final dataset shape: {train.shape}")

Starting memory-optimized feature engineering pipeline...
All features engineered successfully with zero memory fragmentation!
Final dataset shape: (590540, 487)
